# Load the data

In [2]:
import tarfile
import urllib.request
import numpy as np
from pathlib import Path

def extract_contents(movie_files: list[Path]) -> list[str]:
    """Extract movie reviews from a tar file."""
    data = []
    for file in movie_files:
        with open(file, "r") as f:
            movie_review = f.readline()
        data.append(movie_review.strip())
    return data

def load_movies_data() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    tarball_path = Path("datasets/aclImdb_v1.tar.gz")
    if not tarball_path.is_file():
        print("Downloading IMDB Movie Reviews dataset...")
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as movies_tarball:
            movies_tarball.extractall(path="datasets", filter="data")

    train_data = []
    train_pos_files = [ f for f in Path("datasets/aclImdb/train/pos").iterdir() if f.is_file()]
    train_neg_files = [f for f in Path("datasets/aclImdb/train/neg").iterdir() if f.is_file()]
    train_data.extend(extract_contents(train_pos_files))
    train_data.extend(extract_contents(train_neg_files))
    train = np.array(train_data, dtype=object)
    train_labels = np.array([1] * len(train_pos_files) + [0] * len(train_neg_files))

    test_data = []
    test_pos_files = [f for f in Path("datasets/aclImdb/test/pos").iterdir() if f.is_file()]
    test_neg_files = [f for f in Path("datasets/aclImdb/test/neg").iterdir() if f.is_file()]
    test_data.extend(extract_contents(test_pos_files))
    test_data.extend(extract_contents(test_neg_files))
    test = np.array(test_data, dtype=object)
    test_labels = np.array([1] * len(test_pos_files) + [0] * len(test_neg_files))

    return train, test, train_labels, test_labels


In [3]:
X_train, X_test, y_train, y_test = load_movies_data()

In [4]:
X_train.shape

(25000,)

In [5]:
y_train.shape

(25000,)

In [6]:
X_train[10]

"OK. I'm biased. I live near Shrewsbury in England, where this wonderful movie was filmed. It still looks the same now. I remember them filming here quite vividly, and the fake snow on the streets for days on end. Often, when I'm walking through Shrewsbury I see a street or a house and it will remind me of this film.<br /><br />George C. Scott's Scrooge is a more realistic character than many of the other screen versions. His physical appearance isn't the typical miser. Scott's is big and imposing. A man who finds those smaller than himself to be inferior.<br /><br />We all know the story and the quotes. The book is one of the most cherished works in the English language. And I don't believe there are many cynics who would say that people aren't capable of change and redemption. This film version portrays all of that quite beautifully. George C. Scott may be American but he plays the part of the English miser with wonderful skill.<br /><br />I love this movie. If you haven't seen this 

In [7]:
y_train[10]

np.int64(1)

# Create Preprocess Pipeline

In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt")

def preprocess_text(text: str) -> str:
    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove special characters and digits
    text = re.sub(r"[^a-z\s]", "", text)

    # Tokenize
    tokens = text.split()
    # Remove stop words
    stop_words = set(stopwords.words("english"))
    # Keep negations as they are important for sentiment
    negations = {"no", "not", "neither", "nor", "never", "none", "nobody", "nothing", "nowhere"}
    stop_words -= negations
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/olamilekan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/olamilekan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/olamilekan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [23]:
from sklearn.base import BaseEstimator, TransformerMixin

class TextPreprocessingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_transformed = []
        for text in X:
            X_transformed.append(preprocess_text(text))
        return np.array(X_transformed)


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer


preprocess_pipeline = Pipeline([
    ("text_preprocessing", TextPreprocessingTransformer()),
    ("vectorization", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

# Evaluate different models and select the best one

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

scores_lgr = cross_val_score(LogisticRegression(random_state=42), X_train_transformed, y_train, cv=5, scoring="accuracy")
scores_lgr.mean()

np.float64(0.87296)

In [25]:
from sklearn.linear_model import SGDClassifier

scores_sgd = cross_val_score(SGDClassifier(random_state=42), X_train_transformed, y_train, cv=5, scoring="accuracy")
scores_sgd.mean()

np.float64(0.8836)

In [14]:
from sklearn.ensemble import RandomForestClassifier

scores_forest = cross_val_score(RandomForestClassifier(random_state=42), X_train_transformed, y_train, cv=5, scoring="accuracy")
scores_forest.mean()

KeyboardInterrupt: 

In [13]:
from sklearn.naive_bayes import MultinomialNB

scores_multi = cross_val_score(MultinomialNB(), X_train_transformed, y_train, cv=5, scoring="accuracy")
scores_multi.mean()

np.float64(0.8748400000000001)

## I decided to go with `SGDClassifier` since it performs better than the rest in terms of accuracy and training time

# Hyperparameters tuning

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "loss": ["hinge", "log_loss", "modified_huber"],
    "penalty": ["l1", "l2"],
    "learning_rate": ["constant", "optimal"]
}

grid_search = GridSearchCV(SGDClassifier(random_state=42), param_grid, scoring="accuracy", cv=3)
grid_search.fit(X_train_transformed, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SGDClassifier(random_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': ['constant', 'optimal'], 'loss': ['hinge', 'log_loss', ...], 'penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate i

In [17]:
grid_search.best_score_

np.float64(0.8858801004307991)

In [18]:
grid_search.best_params_

{'learning_rate': 'optimal', 'loss': 'modified_huber', 'penalty': 'l2'}

# Train the final model

In [27]:
full_pipeline = Pipeline([
    ("preprocess", preprocess_pipeline),
    ("sgd", SGDClassifier(random_state=42, learning_rate="optimal", loss="modified_huber", penalty="l2")),
])
full_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('sgd', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('text_preprocessing', ...), ('vectorization', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes i

In [28]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

y_pred = full_pipeline.predict(X_test)

print(f"Accuracy score: {accuracy_score(y_test, y_pred)}")
print(f"Precision score: {precision_score(y_test, y_pred)}")
print(f"Recall score: {recall_score(y_test, y_pred)}")

Accuracy score: 0.8812
Precision score: 0.8813830638706579
Recall score: 0.88096


# Save the final model

In [29]:
from shared_lib.model_store import save_model

save_model(full_pipeline, "sgd_classifier")

Model saved to ../models/sgd_classifier.joblib
